# Evals for Agents with Arize

**AI Engineer Europe Workshop — April 7, 2026**

In this notebook we'll:

1. **Set up tracing** — instrument a Claude agent with Phoenix
2. **Build an agent** — a financial analysis chatbot with web search
3. **Look at your data** — error analysis before writing evals
4. **Write a code eval** — a deterministic ticker-mention check
5. **Run built-in evals** — correctness and faithfulness
6. **Write a custom eval rubric** — an actionability eval from scratch
7. **Meta-evaluate** — test the judge against human labels
8. **Run an experiment** — improve the agent and measure the difference

You'll need:
- An Anthropic API key (free at [https://platform.claude.com/settings/keys](https://platform.claude.com/settings/keys))
- A Phoenix Cloud account (free at [app.phoenix.arize.com](https://app.phoenix.arize.com))
  - A Phoenix API key (generate one in Settings)
  - Your Phoenix cloud hostname (looks like https://app.phoenix.arize.com/s/youruser)

## Step 1: Set Up Tracing

Tracing captures a structured record of every step your agent takes — every LLM call, every tool invocation, every decision — with inputs and outputs at each point. We set it up once, and data flows automatically.

In [ ]:
%pip install claude-agent-sdk openinference-instrumentation-claude-agent-sdk arize-phoenix anthropic

In [ ]:
import os
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get('anthropic-api-key')
os.environ["PHOENIX_API_KEY"] = userdata.get('phoenix-api-key')
os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = userdata.get('phoenix-collector-endpoint')

In [ ]:
from phoenix.otel import register

tracer_provider = register(project_name="aie-claude-financial-agent", auto_instrument=True)

/usr/local/lib/python3.12/dist-packages/phoenix/otel/otel.py:433: UserWarning: Could not infer collector endpoint protocol, defaulting to HTTP.
  warnings.warn("Could not infer collector endpoint protocol, defaulting to HTTP.")


🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: aie-claude-financial-agent
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: https://app.phoenix.arize.com/s/lvoss/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {'authorization': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



## Step 2: Build the Agent

We're building a financial analysis chatbot using the Claude Agent SDK. The agent works in two turns:
- **Turn 1: Research** — searches the web for real financial data
- **Turn 2: Write** — compiles the research into a readable report

The `ClaudeSDKClient` maintains conversation context between turns, so the writer has full access to the researcher's findings.

In [ ]:
from claude_agent_sdk import ClaudeSDKClient, ClaudeAgentOptions, AssistantMessage, TextBlock, ResultMessage
from opentelemetry import trace

tracer = trace.get_tracer(__name__)

RESEARCH_PROMPT = """Research {tickers}. Focus on: {focus}.
Use web search to find current financial data, news, and trends."""

WRITE_PROMPT = """Now write a concise financial report based on your research above."""

options = ClaudeAgentOptions(
    model="claude-haiku-4-5-20251001",
    allowed_tools=["WebSearch"],
    permission_mode="acceptEdits",
)


async def financial_report(tickers: str, focus: str, verbose: bool = True) -> str:
    with tracer.start_as_current_span(
        "financial_report",
        attributes={
            "input.value": f"Research: {tickers}\nFocus: {focus}",
        },
    ) as span:
        async with ClaudeSDKClient(options=options) as client:
            # Turn 1: Research
            if verbose:
                print(f"--- Researching {tickers} ({focus}) ---")
            await client.query(RESEARCH_PROMPT.format(tickers=tickers, focus=focus))
            async for message in client.receive_response():
                if verbose and isinstance(message, AssistantMessage):
                    for block in message.content:
                        if isinstance(block, TextBlock):
                            preview = block.text[:200].replace("\n", " ")
                            print(f"  [research] {preview}...")

            # Turn 2: Write report
            if verbose:
                print(f"--- Writing report ---")
            await client.query(WRITE_PROMPT)
            report = ""
            async for message in client.receive_response():
                if isinstance(message, AssistantMessage):
                    for block in message.content:
                        if isinstance(block, TextBlock):
                            report += block.text
                            if verbose:
                                preview = block.text[:200].replace("\n", " ")
                                print(f"  [writing] {preview}...")
                elif isinstance(message, ResultMessage):
                    report = message.result or report

            span.set_attribute("output.value", report)
            return report

### Run the agent once to test

This will take a minute or two

In [ ]:
result = await financial_report("TSLA", "financial performance and growth outlook")
print(result)

--- Researching TSLA (financial performance and growth outlook) ---
  [research] I'll search for current financial data and growth outlook information on Tesla....
  [research] ## Tesla (TSLA) Research: Financial Performance & Growth Outlook  ### **Financial Performance Summary**  **2025 Results - Mixed Performance:** - **Revenue**: $94.8B (down 2.93% YoY) - **first annual r...
--- Writing report ---
  [writing] # TESLA, INC. (NASDAQ: TSLA) – FINANCIAL REPORT ## March 2026  ---  ## EXECUTIVE SUMMARY  Tesla faces a critical transition period in 2026. While the company maintains industry leadership and strong g...
# TESLA, INC. (NASDAQ: TSLA) – FINANCIAL REPORT
## March 2026

---

## EXECUTIVE SUMMARY

Tesla faces a critical transition period in 2026. While the company maintains industry leadership and strong growth prospects in energy storage and robotics, core automotive performance has deteriorated significantly. 2025 marked the first annual revenue decline in company history, with pr

## Step 3: Generate Test Data

To run meaningful evals, we need more than one trace. Here are 12 test queries covering different tickers, focus areas, and levels of complexity.

In [ ]:
test_queries = [
    {"tickers": "AAPL", "focus": "revenue growth and services segment"},
    {"tickers": "NVDA", "focus": "AI chip demand and valuation metrics"},
    {"tickers": "AMZN", "focus": "AWS performance and profitability"},
    {"tickers": "GOOGL", "focus": "advertising revenue and AI strategy"},
    {"tickers": "MSFT", "focus": "cloud computing segment"},
    {"tickers": "META", "focus": "metaverse investments and ad revenue"},
    {"tickers": "TSLA", "focus": "vehicle deliveries and margins"},
    {"tickers": "RIVN", "focus": "financial health and future growth"},
    {"tickers": "AAPL, MSFT", "focus": "comparative financial analysis"},
    {"tickers": "NVDA", "focus": "competitive landscape and market share"},
    {"tickers": "KO", "focus": "dividend yield and stability"},
    {"tickers": "AMZN", "focus": "profitability trends and outlook"},
]

In [ ]:
for i, query in enumerate(test_queries):
    print(f"[{i+1}/{len(test_queries)}] {query['tickers']} — {query['focus']}")
    await financial_report(query["tickers"], query["focus"], verbose=False)
    print(f"  done")

[1/12] AAPL — revenue growth and services segment
  done
[2/12] NVDA — AI chip demand and valuation metrics
  done
[3/12] AMZN — AWS performance and profitability
  done
[4/12] GOOGL — advertising revenue and AI strategy
  done
[5/12] MSFT — cloud computing segment
  done
[6/12] META — metaverse investments and ad revenue
  done
[7/12] TSLA — vehicle deliveries and margins
  done
[8/12] RIVN — financial health and future growth
  done
[9/12] AAPL, MSFT — comparative financial analysis
  done
[10/12] NVDA — competitive landscape and market share
  done
[11/12] KO — dividend yield and stability
  done
[12/12] AMZN — profitability trends and outlook
  done


## Step 3.5: Look at Your Data — Error Analysis

Before writing a single eval, read your traces. This is the step most tutorials skip — and it's the most important one. You need to understand *what's actually going wrong* before you can measure it.

We'll:
1. **Examine traces** — read the input and output of a few runs
2. **Categorize failures** — label each trace with a root cause
3. **Build a frequency table** — see where to focus first

### Categorize failures by root cause

Now label each trace with what you observe. Don't overthink the categories — "looks good," "hallucination," "reasoning gap," "missing data" are fine. The goal is to build intuition about *what kinds of things go wrong*.

In [ ]:
import pandas as pd

# Label each trace with a root cause category.
# Replace these with your own observations after reading the traces above!
trace_categories = {
    "TSLA — financial performance": "looks good",
    "NVDA — AI chip demand": "looks good",
    "AMZN — AWS performance": "possible hallucination",
    "GOOGL — advertising revenue": "looks good",
    "MSFT — cloud computing": "looks good",
    "META — metaverse investments": "reasoning gap",
    "TSLA — vehicle deliveries": "looks good",
    "RIVN — financial health": "unverifiable data",
    "AAPL, MSFT — comparative": "reasoning gap",
    "NVDA — competitive landscape": "possible hallucination",
    "KO — dividend yield": "missing recommendation",
    "AMZN — profitability trends": "possible hallucination",
}

categories_df = pd.DataFrame(
    list(trace_categories.items()),
    columns=["trace", "category"]
)
display(categories_df)

,trace,category
0,TSLA — financial performance,looks good
1,NVDA — AI chip demand,looks good
2,AMZN — AWS performance,possible hallucination
3,GOOGL — advertising revenue,looks good
4,MSFT — cloud computing,looks good
5,META — metaverse investments,reasoning gap
6,TSLA — vehicle deliveries,looks good
7,RIVN — financial health,unverifiable data
8,"AAPL, MSFT — comparative",reasoning gap
9,NVDA — competitive landscape,possible hallucination


### Frequency table — where should we focus?

Count the categories. The most frequent failure type is where you focus first. But frequency isn't everything — **frequency × severity = priority**. A hallucinated stock price is worse than an awkward sentence.

In [ ]:
# Frequency table of root cause categories
print("Root cause frequency:")
print("-" * 35)
counts = categories_df["category"].value_counts()
for category, count in counts.items():
    bar = "█" * count
    print(f"  {category:<25} {count}  {bar}")
print(f"\nTotal traces: {len(categories_df)}")
print(f"Failure rate: {(len(categories_df) - counts.get('looks good', 0)) / len(categories_df):.0%}")

Root cause frequency:
-----------------------------------
  looks good                5  █████
  possible hallucination    3  ███
  reasoning gap             2  ██
  unverifiable data         1  █
  missing recommendation    1  █

Total traces: 12
Failure rate: 58%


## Step 4: Code Eval — Ticker Mention Check

Code evals are deterministic functions — no LLM, no API call, no cost. Our simplest useful check: does the output actually mention the ticker we asked about?


Fetch your data from Phoenix

In [ ]:
from phoenix.client import Client

px_client = Client()
spans_df = px_client.spans.get_spans_dataframe(project_name="aie-claude-financial-agent")

# Filter to top-level agent spans (the full financial_report calls)
parent_spans = spans_df[spans_df["parent_id"].isna()].copy()
parent_spans.rename(columns={"attributes.input.value": "input"}, inplace=True)
parent_spans.rename(columns={"attributes.output.value": "output"}, inplace=True)
print(f"Found {len(parent_spans)} top-level spans")

Found 13 top-level spans


In [ ]:
print(f"Using {len(parent_spans)} top-level spans for code evals")

Using 13 top-level spans for code evals


In [ ]:
import re
from phoenix.evals import create_evaluator
from phoenix.trace import suppress_tracing
from phoenix.evals import evaluate_dataframe


@create_evaluator(name="mentions_ticker", kind="code")
def mentions_ticker(input, output):
    """Code eval: does the output mention the ticker(s) we asked about?"""
    # Extract ticker symbols from the input (uppercase 1-5 letter words)
    tickers = re.findall(r"\b([A-Z]{1,5})\b", input)
    # Filter to likely tickers (skip common words)
    likely_tickers = [
        t
        for t in tickers
        if len(t) >= 2
        and t not in ("AI", "US", "CEO", "CFO", "IPO", "ETF", "AWS", "USE")
    ]

    if not likely_tickers or not output:
        return {"label": "unknown", "score": 0}

    missing = [t for t in likely_tickers if t not in output.upper()]

    if not missing:
        return {"label": "pass", "score": 1}
    else:
        return {
            "label": "fail",
            "score": 0,
            "explanation": f"Missing tickers: {', '.join(missing)}",
        }


with suppress_tracing():
    results = evaluate_dataframe(dataframe=parent_spans, evaluators=[mentions_ticker])

Evaluating Dataframe |          | 0/13 (0.0%) | ⏳ 00:00<? | ?it/s

In [ ]:
# Show all mentions_ticker results (not just head)
import pandas as pd

ticker_scores = pd.json_normalize(results["mentions_ticker_score"])
print("Mentions Ticker Results:")
print(f"  {ticker_scores['label'].value_counts().to_dict()}")
print(f"  {ticker_scores['label'].value_counts().get('pass', 0)}/{len(ticker_scores)} passed")

# Show any failures with explanations
failures = ticker_scores[ticker_scores["label"] == "fail"]
if len(failures) > 0:
    print(f"\nFailures:")
    for _, row in failures.iterrows():
        print(f"  {row.get('explanation', 'no explanation')}")
else:
    print("\nAll passed!")

Mentions Ticker Results:
  {'pass': 11, 'fail': 2}
  11/13 passed

Failures:
  Missing tickers: TSLA
  Missing tickers: AMZN


Send the results back to Phoenix

In [ ]:
from phoenix.evals.utils import to_annotation_dataframe

# Log mentions_ticker results back to Phoenix
mentions_ticker_evaluations = to_annotation_dataframe(dataframe=results)
Client().spans.log_span_annotations_dataframe(dataframe=mentions_ticker_evaluations)

## Step 5: Built-In Eval — Correctness

Phoenix ships with pre-built evaluators for common checks. The **Correctness** evaluator uses an LLM judge to assess whether a response is factually accurate, complete, and logically consistent — no prompt engineering required.

In [ ]:
from phoenix.evals import LLM
from phoenix.evals.metrics import CorrectnessEvaluator
from phoenix.trace import suppress_tracing
import anthropic

llm = LLM(model="claude-sonnet-4-6", provider="anthropic")

# Built-in evaluator — no prompt template needed
correctness_eval = CorrectnessEvaluator(llm=llm)

We suppress tracing here so it doesn't trace our evaluator itself.

In [ ]:
with suppress_tracing():
    correctness_results = evaluate_dataframe(
        dataframe=parent_spans, evaluators=[correctness_eval]
    )

correctness_results.head()

Evaluating Dataframe |          | 0/13 (0.0%) | ⏳ 00:00<? | ?it/s

,name,span_kind,parent_id,start_time,end_time,status_code,status_message,events,context.span_id,context.trace_id,...,attributes.llm.token_count.prompt,attributes.llm.token_count.prompt_details.cache_read,attributes.llm.token_count.completion,attributes.llm.token_count.prompt_details.cache_write,attributes.llm.model_name,attributes.output.mime_type,output,attributes.llm.system,correctness_execution_details,correctness_score
context.span_id,,,,,,,,,,,,,,,,,,,,,
6d6dbe1140d275f4,financial_report,UNKNOWN,None,2026-03-21 21:11:54.738412+00:00,2026-03-21 21:12:51.370389+00:00,UNSET,,[],6d6dbe1140d275f4,9525bc92a947767782d5131a7e62d872,...,NaN,NaN,NaN,NaN,None,None,Here's a concise financial report based on the...,None,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'correctness', 'score': 0.0, 'label':..."
32f94538037fbe62,financial_report,UNKNOWN,None,2026-03-21 21:11:13.240006+00:00,2026-03-21 21:11:54.692825+00:00,UNSET,,[],32f94538037fbe62,14bff5d459bf77340306b6510c2a0ede,...,NaN,NaN,NaN,NaN,None,None,# THE COCA-COLA COMPANY (KO)\n## Financial Rep...,None,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'correctness', 'score': 0.0, 'label':..."
b022c01a860c2495,financial_report,UNKNOWN,None,2026-03-21 21:10:06.310924+00:00,2026-03-21 21:11:13.193977+00:00,UNSET,,[],b022c01a860c2495,3a35879662ca25847abe12ec60f46fde,...,NaN,NaN,NaN,NaN,None,None,# NVIDIA CORPORATION (NVDA)\n## Financial Repo...,None,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'correctness', 'score': 0.0, 'label':..."
318187f1e74c2248,financial_report,UNKNOWN,None,2026-03-21 21:08:40.159986+00:00,2026-03-21 21:10:06.261073+00:00,UNSET,,[],318187f1e74c2248,31367f66ad0a75ca2ec89de626eb4a36,...,NaN,NaN,NaN,NaN,None,None,I'll present the concise financial report here...,None,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'correctness', 'score': 0.0, 'label':..."
03bd130b4af20494,financial_report,UNKNOWN,None,2026-03-21 21:07:58.941278+00:00,2026-03-21 21:08:40.116730+00:00,UNSET,,[],03bd130b4af20494,377b9ffbf2590ce0a6cd26ff685d08f6,...,NaN,NaN,NaN,NaN,None,None,"# RIVIAN AUTOMOTIVE, INC. (NASDAQ: RIVN)\n## F...",None,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'correctness', 'score': 0.0, 'label':..."


And send the results to Phoenix again.

In [ ]:
from phoenix.evals.utils import to_annotation_dataframe

correctness_annotations = to_annotation_dataframe(dataframe=correctness_results)
Client().spans.log_span_annotations_dataframe(dataframe=correctness_annotations)

## Step 5.5: Built-In Eval — Faithfulness

Correctness gave us 0/13 — not because the reports are bad, but because the judge is trying to fact-check real-time financial data against its training data. That's the wrong question.

**Faithfulness** asks a better question: does the report stick to the source material the agent actually found? Instead of fact-checking against the whole internet, it checks whether the output is grounded in the research from Turn 1.

The key difference: we give the judge the agent's research as **context**.

### Extract research context from traces

Our agent works in two turns: research (web search), then write (report). The research turn's output is the context we need. We extract it from child spans and attach it to our parent spans.

In [ ]:
# Extract context: the research output from Turn 1 (child spans of our parent spans)
# Child LLM spans contain the research the agent gathered before writing

child_spans = spans_df[spans_df["parent_id"].notna()].copy()

# Get the first assistant response per trace (that's the research turn)
# Group by trace_id and take the first LLM output
research_context = {}
for trace_id, group in child_spans.groupby("context.trace_id"):
    # Find spans with LLM output (the research response)
    llm_spans = group[group["attributes.llm.output_messages"].notna()]
    if len(llm_spans) > 0:
        # Take the first LLM response as research context
        first_response = llm_spans.iloc[0]
        output = first_response.get("attributes.output.value", "")
        if output:
            research_context[trace_id] = str(output)[:5000]  # Truncate for token limits

# Map trace_id to parent span and add context column
parent_spans["context"] = parent_spans["context.trace_id"].map(research_context)
print(f"Added context to {parent_spans['context'].notna().sum()}/{len(parent_spans)} spans")
print(f"\nSample context (first 200 chars):")
first_ctx = parent_spans["context"].dropna().iloc[0] if parent_spans["context"].notna().any() else "N/A"
print(f"  {first_ctx[:200]}...")

Added context to 13/13 spans

Sample context (first 200 chars):
  Here's a concise financial report based on the research:

---

# **AMAZON.COM INC. (NASDAQ: AMZN)**
## **Financial Analysis Report – March 2026**

---

## **EXECUTIVE SUMMARY**

Amazon demonstrated ex...


### Run the faithfulness evaluator

FaithfulnessEvaluator needs three columns: `input`, `output`, and `context`. We have all three now.

In [ ]:
from phoenix.evals.metrics import FaithfulnessEvaluator

faithfulness_eval = FaithfulnessEvaluator(llm=llm)

# Only run on spans that have context
spans_with_context = parent_spans[parent_spans["context"].notna()].copy()

with suppress_tracing():
    faith_results = evaluate_dataframe(
        dataframe=spans_with_context,
        evaluators=[faithfulness_eval]
    )

faith_results.head()

Evaluating Dataframe |          | 0/13 (0.0%) | ⏳ 00:00<? | ?it/s

,name,span_kind,parent_id,start_time,end_time,status_code,status_message,events,context.span_id,context.trace_id,...,attributes.llm.token_count.prompt_details.cache_read,attributes.llm.token_count.completion,attributes.llm.token_count.prompt_details.cache_write,attributes.llm.model_name,attributes.output.mime_type,output,attributes.llm.system,context,faithfulness_execution_details,faithfulness_score
context.span_id,,,,,,,,,,,,,,,,,,,,,
6d6dbe1140d275f4,financial_report,UNKNOWN,None,2026-03-21 21:11:54.738412+00:00,2026-03-21 21:12:51.370389+00:00,UNSET,,[],6d6dbe1140d275f4,9525bc92a947767782d5131a7e62d872,...,NaN,NaN,NaN,None,None,Here's a concise financial report based on the...,None,Here's a concise financial report based on the...,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'faithfulness', 'score': 1.0, 'label'..."
32f94538037fbe62,financial_report,UNKNOWN,None,2026-03-21 21:11:13.240006+00:00,2026-03-21 21:11:54.692825+00:00,UNSET,,[],32f94538037fbe62,14bff5d459bf77340306b6510c2a0ede,...,NaN,NaN,NaN,None,None,# THE COCA-COLA COMPANY (KO)\n## Financial Rep...,None,# THE COCA-COLA COMPANY (KO)\n## Financial Rep...,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'faithfulness', 'score': 1.0, 'label'..."
b022c01a860c2495,financial_report,UNKNOWN,None,2026-03-21 21:10:06.310924+00:00,2026-03-21 21:11:13.193977+00:00,UNSET,,[],b022c01a860c2495,3a35879662ca25847abe12ec60f46fde,...,NaN,NaN,NaN,None,None,# NVIDIA CORPORATION (NVDA)\n## Financial Repo...,None,# NVIDIA CORPORATION (NVDA)\n## Financial Repo...,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'faithfulness', 'score': 1.0, 'label'..."
318187f1e74c2248,financial_report,UNKNOWN,None,2026-03-21 21:08:40.159986+00:00,2026-03-21 21:10:06.261073+00:00,UNSET,,[],318187f1e74c2248,31367f66ad0a75ca2ec89de626eb4a36,...,NaN,NaN,NaN,None,None,I'll present the concise financial report here...,None,I'll present the concise financial report here...,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'faithfulness', 'score': 1.0, 'label'..."
03bd130b4af20494,financial_report,UNKNOWN,None,2026-03-21 21:07:58.941278+00:00,2026-03-21 21:08:40.116730+00:00,UNSET,,[],03bd130b4af20494,377b9ffbf2590ce0a6cd26ff685d08f6,...,NaN,NaN,NaN,None,None,"# RIVIAN AUTOMOTIVE, INC. (NASDAQ: RIVN)\n## F...",None,"# RIVIAN AUTOMOTIVE, INC. (NASDAQ: RIVN)\n## F...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'faithfulness', 'score': 1.0, 'label'..."


In [ ]:
# Compare correctness vs faithfulness results
import pandas as pd

faith_scores = pd.json_normalize(faith_results["faithfulness_score"])
print("Faithfulness results:")
print(f"  {faith_scores['label'].value_counts().to_dict()}")
print(f"\nCorrectness gave us 0/13 passes — not useful for real-time data.")
print(f"Faithfulness gives us a meaningful split — choosing the right eval matters more than tuning it.")

Faithfulness results:
  {'faithful': 13}

Correctness gave us 0/13 passes — not useful for real-time data.
Faithfulness gives us a meaningful split — choosing the right eval matters more than tuning it.


In [ ]:
# Log faithfulness results back to Phoenix
faith_annotations = to_annotation_dataframe(dataframe=faith_results)
Client().spans.log_span_annotations_dataframe(dataframe=faith_annotations)
print("Faithfulness annotations logged to Phoenix")

Faithfulness annotations logged to Phoenix


## Step 6: Custom Eval Rubric — Actionability

Built-in evals are general-purpose. Custom rubrics check what matters for *your* application. Our financial analyst should produce actionable recommendations — not just summarize data, but tell the reader what to do.

A good eval prompt has five parts:
1. **Define the judge's role** — domain context
2. **Explicit criteria** — what passes, what fails
3. **Present the data clearly** — labels and delimiters
4. **Add labeled examples** — show, don't just tell
5. **Constrain the output** — binary labels, not numeric scales

In [ ]:
actionability_template = """
You are an expert financial analyst evaluator. Your task is to judge whether
a financial report provides actionable investment guidance, not just raw data.

ACTIONABLE — The report:
- Contains specific recommendations (buy/sell/hold or equivalent guidance)
- Identifies concrete risks with supporting data
- Includes forward-looking analysis, not just historical data
- Provides context for WHY recommendations are made

NOT ACTIONABLE — The report:
- Only summarizes publicly available data without interpretation
- Lacks specific recommendations or next steps
- Presents risks without supporting evidence
- Contains only backward-looking analysis

Here are examples of each:

Example — ACTIONABLE:
\"Based on NVDA's 122% YoY revenue growth driven by data center demand,
strong forward P/E of 35x relative to sector median of 22x, and expanding
margins, NVDA presents a compelling growth position. Key risk: concentration
in AI training chips (~70% of revenue). Recommendation: accumulate on
pullbacks below $800.\"

Example — NOT ACTIONABLE:
\"NVDA is a major player in the semiconductor industry. The company has seen
significant growth in recent years driven by AI demand. NVDA's stock has
performed well. Investors should consider various factors when making
investment decisions.\"

[BEGIN DATA]
************
User query: {input}
************
Financial Report: {output}
************
[END DATA]

Based on the criteria above, is this financial report ACTIONABLE or NOT ACTIONABLE?
"""

Run our new custom judge.

In [ ]:
from phoenix.evals import ClassificationEvaluator

actionability_evaluator = ClassificationEvaluator(
    name="actionability",
    prompt_template=actionability_template,
    llm=llm,
    choices={"actionable": 1.0, "not actionable": 0.0},
)

with suppress_tracing():
    action_results_df = evaluate_dataframe(
        dataframe=parent_spans, evaluators=[actionability_evaluator]
    )

Evaluating Dataframe |          | 0/13 (0.0%) | ⏳ 00:00<? | ?it/s

In [ ]:
import pandas as pd

pd.json_normalize(action_results_df["actionability_score"].head(10))

,name,score,label,explanation,kind,direction,metadata.model
0,actionability,1.0,actionable,This financial report is clearly actionable. I...,llm,maximize,claude-sonnet-4-6
1,actionability,1.0,actionable,The report meets all four criteria for being a...,llm,maximize,claude-sonnet-4-6
2,actionability,1.0,actionable,The report qualifies as actionable for several...,llm,maximize,claude-sonnet-4-6
3,actionability,1.0,actionable,The report provides several elements that qual...,llm,maximize,claude-sonnet-4-6
4,actionability,1.0,actionable,This report is clearly actionable for several ...,llm,maximize,claude-sonnet-4-6
5,actionability,0.0,not actionable,"The financial report, as described, lacks a co...",llm,maximize,claude-sonnet-4-6
6,actionability,0.0,not actionable,The report presents strong financial data and ...,llm,maximize,claude-sonnet-4-6
7,actionability,1.0,actionable,The report meets all four criteria for being a...,llm,maximize,claude-sonnet-4-6
8,actionability,0.0,not actionable,The report provides substantial forward-lookin...,llm,maximize,claude-sonnet-4-6
9,actionability,0.0,not actionable,"The report is data-rich and well-structured, p...",llm,maximize,claude-sonnet-4-6


Log actionability results back to Phoenix


In [ ]:
action_evaluations = to_annotation_dataframe(dataframe=action_results_df)
Client().spans.log_span_annotations_dataframe(dataframe=action_evaluations)

## Step 6.5: Meta-Evaluation — Can You Trust Your Judge?

Your LLM judge is a classifier. It makes predictions (actionable / not actionable) that you can check against ground truth — your own human judgment. This is **meta-evaluation**: testing the test.

We'll:
1. **Hand-label a few examples** — you read the output and decide
2. **Run the judge on the same examples** — see what it says
3. **Compare** — where do they agree? Where do they disagree?
4. **Calculate precision and recall** — put numbers on judge quality

Hand-label the examples in the Phoenix UI, then fetch them down to work with them.

In [ ]:
from phoenix.client.types.spans import SpanQuery

client = Client()

query = SpanQuery().where("annotations['human_actionable']") # .label == 'not actionable' if you wanted to filter

human_spans = client.spans.get_spans_dataframe(
    query=query,
    project_identifier="aie-claude-financial-agent",
)

human_spans.rename(columns={"attributes.input.value": "input"}, inplace=True)
human_spans.rename(columns={"attributes.output.value": "output"}, inplace=True)

Now get the actual annotations and join them

In [ ]:
annotations = client.spans.get_span_annotations_dataframe(
    spans_dataframe=human_spans,
    project_identifier="aie-claude-financial-agent",
    include_annotation_names=["human_actionable"],
)

annotated_spans = annotations.join(human_spans, how="left")

### Step 2: Run the judge on the same examples

Now run the actionability evaluator on exactly the same traces. Same inputs, same outputs — the only question is whether the judge agrees with you.

In [ ]:
# Run our actionability judge on the labeled subset
with suppress_tracing():
    judge_results = evaluate_dataframe(
        dataframe=human_spans, evaluators=[actionability_evaluator]
    )

# Extract judge labels
judge_labels = pd.json_normalize(judge_results["actionability_score"])
annotated_spans["judge_label"] = judge_labels["label"].values
annotated_spans["judge_explanation"] = judge_labels["explanation"].values

Evaluating Dataframe |          | 0/6 (0.0%) | ⏳ 00:00<? | ?it/s

In [ ]:
display(annotated_spans)

,id,created_at,updated_at,source,user_id,annotation_name,annotator_kind,metadata,identifier,result.label,...,end_time,status_code,status_message,events,context.span_id,context.trace_id,output,input,judge_label,judge_explanation
span_id,,,,,,,,,,,,,,,,,,,,,
03bd130b4af20494,U3BhbkFubm90YXRpb246MTY3OQ==,2026-03-21T23:16:22.715771+00:00,2026-03-21T23:16:22.715771+00:00,APP,VXNlcjoz,human_actionable,HUMAN,{},px-app:VXNlcjoz,actionable,...,2026-03-21 21:08:40.116730+00:00,UNSET,,[],03bd130b4af20494,377b9ffbf2590ce0a6cd26ff685d08f6,"# RIVIAN AUTOMOTIVE, INC. (NASDAQ: RIVN)\n## F...",Research: RIVN\nFocus: financial health and fu...,actionable,The report clearly meets all actionable criter...
318187f1e74c2248,U3BhbkFubm90YXRpb246MTY3OA==,2026-03-21T23:16:19.363866+00:00,2026-03-21T23:16:19.363866+00:00,APP,VXNlcjoz,human_actionable,HUMAN,{},px-app:VXNlcjoz,not actionable,...,2026-03-21 21:10:06.261073+00:00,UNSET,,[],318187f1e74c2248,31367f66ad0a75ca2ec89de626eb4a36,I'll present the concise financial report here...,"Research: AAPL, MSFT\nFocus: comparative finan...",actionable,The report meets all four criteria for being a...
32f94538037fbe62,U3BhbkFubm90YXRpb246MTY3Nw==,2026-03-21T23:16:13.577160+00:00,2026-03-21T23:16:13.577160+00:00,APP,VXNlcjoz,human_actionable,HUMAN,{},px-app:VXNlcjoz,actionable,...,2026-03-21 21:11:54.692825+00:00,UNSET,,[],32f94538037fbe62,14bff5d459bf77340306b6510c2a0ede,# THE COCA-COLA COMPANY (KO)\n## Financial Rep...,Research: KO\nFocus: dividend yield and stability,actionable,This report is actionable for several reasons:...
6d6dbe1140d275f4,U3BhbkFubm90YXRpb246MTY3Ng==,2026-03-21T23:16:08.722567+00:00,2026-03-21T23:16:08.722567+00:00,APP,VXNlcjoz,human_actionable,HUMAN,{},px-app:VXNlcjoz,actionable,...,2026-03-21 21:12:51.370389+00:00,UNSET,,[],6d6dbe1140d275f4,9525bc92a947767782d5131a7e62d872,Here's a concise financial report based on the...,Research: AMZN\nFocus: profitability trends an...,actionable,This report qualifies as actionable for severa...
b022c01a860c2495,U3BhbkFubm90YXRpb246MTY3NQ==,2026-03-21T23:07:12.801899+00:00,2026-03-21T23:07:12.801899+00:00,APP,VXNlcjoz,human_actionable,HUMAN,{},px-app:VXNlcjoz,not actionable,...,2026-03-21 21:11:13.193977+00:00,UNSET,,[],b022c01a860c2495,3a35879662ca25847abe12ec60f46fde,# NVIDIA CORPORATION (NVDA)\n## Financial Repo...,Research: NVDA\nFocus: competitive landscape a...,actionable,This report meets the criteria for actionable ...
fb5a8e97b5c675d1,U3BhbkFubm90YXRpb246MTY3NA==,2026-03-21T23:05:47.591119+00:00,2026-03-21T23:06:06.119470+00:00,APP,VXNlcjoz,human_actionable,HUMAN,{},px-app:VXNlcjoz,not actionable,...,2026-03-21 21:03:40.461114+00:00,UNSET,,[],fb5a8e97b5c675d1,c42f9697dcba2e708b67e0261c1c68ee,# NVIDIA CORPORATION (NVDA) - FINANCIAL REPORT...,Research: NVDA\nFocus: AI chip demand and valu...,not actionable,"The report contains rich, data-backed content ..."


### Step 3: Compare — where do they agree and disagree?

When the judge disagrees with you, read its explanation. Often it reveals ambiguity in your rubric — that's the most valuable insight from meta-evaluation.

In [ ]:
# Side-by-side comparison
comparison = annotated_spans[["input", "result.label", "judge_label"]].copy()
comparison["agree"] = comparison["result.label"] == comparison["judge_label"]

display(comparison)

,input,result.label,judge_label,agree
span_id,,,,
03bd130b4af20494,Research: RIVN\nFocus: financial health and fu...,actionable,actionable,True
318187f1e74c2248,"Research: AAPL, MSFT\nFocus: comparative finan...",not actionable,actionable,False
32f94538037fbe62,Research: KO\nFocus: dividend yield and stability,actionable,actionable,True
6d6dbe1140d275f4,Research: AMZN\nFocus: profitability trends an...,actionable,actionable,True
b022c01a860c2495,Research: NVDA\nFocus: competitive landscape a...,not actionable,actionable,False
fb5a8e97b5c675d1,Research: NVDA\nFocus: AI chip demand and valu...,not actionable,not actionable,True


### Step 4: Precision and recall

- **Precision**: When the judge says "not actionable," is it right?
- **Recall**: Of all reports that are truly not actionable, how many does the judge catch?

In most eval scenarios, prioritize **recall** — it's better to flag false positives than to miss real failures.

In [ ]:
# Simple precision/recall calculation for "not actionable"

human_fail = set(comparison[comparison["result.label"] == "not actionable"].index)
judge_fail = set(comparison[comparison["judge_label"] == "not actionable"].index)

true_positives = len(human_fail & judge_fail)   # Both agree it's not actionable
false_positives = len(judge_fail - human_fail)   # Judge says fail, human says pass
false_negatives = len(human_fail - judge_fail)   # Human says fail, judge says pass
true_negatives = len(comparison) - true_positives - false_positives - false_negatives

print("Confusion Matrix")
print("=" * 45)
print(f"                    Judge: fail  Judge: pass")
print(f"  Human: fail           {true_positives}            {false_negatives}")
print(f"  Human: pass           {false_positives}            {true_negatives}")
print()

precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0

print(f"Precision: {precision:.0%}  (when the judge says 'fail', is it right?)")
print(f"Recall:    {recall:.0%}  (of all real fails, how many does it catch?)")
print(f"\nWith only {len(comparison)} examples, these numbers are rough.")
print(f"For a proper validation, aim for 20-50 labeled examples.")

Confusion Matrix
                    Judge: fail  Judge: pass
  Human: fail           1            2
  Human: pass           0            3

Precision: 100%  (when the judge says 'fail', is it right?)
Recall:    33%  (of all real fails, how many does it catch?)

With only 6 examples, these numbers are rough.
For a proper validation, aim for 20-50 labeled examples.


## Step 7: Improve the Agent and Run an Experiment

Evals told us what's wrong. Now we fix it — and *prove* it's better by running the same inputs through the improved agent and comparing scores side by side.

**The process:**
1. Save failing traces as a dataset (do this in the Phoenix UI)
2. Update the agent's prompts based on eval feedback
3. Run an experiment comparing old vs. new

In [ ]:
# Updated prompts — explicitly request the things our evals found missing

IMPROVED_RESEARCH_PROMPT = """Research {tickers}. Focus on: {focus}.
Use web search to find current financial data, news, and trends.

You MUST include:
- Specific financial ratios (P/E, P/B, debt-to-equity, ROE)
- News from the last 6 months
- Current stock price or recent performance data
- Competitive context and market positioning"""

IMPROVED_WRITE_PROMPT = """Now write a concise financial report based on your research above.

The report MUST be actionable. Specifically:
- Include explicit buy/sell/hold recommendations with supporting reasoning
- Identify concrete risks with supporting data
- Include forward-looking analysis, not just historical data
- Provide context for WHY each recommendation is made
- If there are multiple tickers, include a dedicated comparison section"""


async def improved_financial_report(tickers: str, focus: str) -> str:
    with tracer.start_as_current_span(
        "financial_report",
        attributes={
            "input.value": f"Research: {tickers}\nFocus: {focus}",
        },
    ) as span:
        async with ClaudeSDKClient(options=options) as client:
            # Turn 1: Research (improved prompt)
            await client.query(
                IMPROVED_RESEARCH_PROMPT.format(tickers=tickers, focus=focus)
            )
            async for message in client.receive_response():
                pass  # Let the research complete

            # Turn 2: Write report (improved prompt)
            await client.query(IMPROVED_WRITE_PROMPT)
            report = ""
            async for message in client.receive_response():
                if isinstance(message, AssistantMessage):
                    for block in message.content:
                        if isinstance(block, TextBlock):
                            report += block.text
                elif isinstance(message, ResultMessage):
                    report = message.result or report

            span.set_attribute("output.value", report)
            return report

Create a task - this is what the experiment is testing; in this case, the full agent. But we could be testing an individual tool call or something smaller.

In [ ]:
async def my_task(example):
    # Dataset examples from Phoenix nest the span input: {"input": "Research: TSLA\nFocus: ..."}
    inp = example.input
    if isinstance(inp, dict) and "input" in inp:
        inp = inp["input"]

    # Parse "Research: TSLA\nFocus: vehicle deliveries and margins"
    lines = inp.split("\n")
    tickers = lines[0].replace("Research: ", "").strip()
    focus = lines[1].replace("Focus: ", "").strip() if len(lines) > 1 else ""

    return await improved_financial_report(tickers, focus)

Set up our experiment evaluators to correctly handle the dataset we'll be pulling down.

In [ ]:
from phoenix.evals import ClassificationEvaluator

actionability_exp_evaluator = ClassificationEvaluator(
    name="actionability",
    prompt_template=actionability_template,
    llm=llm,
    choices={"actionable": 1.0, "not actionable": 0.0},
)

actionability_exp_evaluator.bind(
    {
        "input": lambda x: x["input"]["input"] if isinstance(x["input"], dict) else x["input"],
        "output": "output",
    }
)

evaluators = [actionability_exp_evaluator]

Now pull down our dataset of failing examples

In [ ]:
# Retrieve the dataset you saved from the Phoenix UI
# (filter traces by failing evals -> Save as Dataset -> name it "datacamp-financial-agent-fails")
dataset = Client().datasets.get_dataset(dataset="AIE financial agent failures")

display(dataset)

Dataset(id='RGF0YXNldDo0Mg==', name='AIE financial agent failures', version_id='RGF0YXNldFZlcnNpb246NTg=', examples=6)

And run the experiment!

In [ ]:
from phoenix.client import AsyncClient

async_client = AsyncClient()
experiment = await async_client.experiments.run_experiment(
    dataset=dataset, task=my_task, evaluators=evaluators, timeout=300
)

🧪 Experiment started.
📺 View dataset experiments: https://app.phoenix.arize.com/s/lvoss/datasets/RGF0YXNldDo0Mg==/experiments
🔗 View this experiment: https://app.phoenix.arize.com/s/lvoss/datasets/RGF0YXNldDo0Mg==/compare?experimentId=RXhwZXJpbWVudDoxMDI=


running tasks |          | 0/6 (0.0%) | ⏳ 00:00<? | ?it/s

✅ Task runs completed.
🧠 Evaluation started.


running experiment evaluations |          | 0/6 (0.0%) | ⏳ 00:00<? | ?it/s

Experiment completed: 6 task runs, 1 evaluator runs, 6 evaluations
